# ClonalOrigin sequential model and its summary statistics for SBI

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import Phylo
import torch
from sbi.inference import NPE_C
from sbi.analysis import plot_summary
from sbi.diagnostics import run_sbc, check_sbc
from sbi.analysis.plot import sbc_rank_plot
import sys
sys.path.append('../pysimARG')
from clonal_genealogy import ClonalTree
from newick_to_tree import newick_to_tree
from ClonalOrigin_seq_sim import ClonalOrigin_seq_sim
from homoplasy_index import homoplasy_index
from G4_test import G4_test
from LD_r import LD_r

torch_device = "cpu"

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


## Load simulation data

Load genome data and clonal tree.

In [2]:
genome_simbac = np.loadtxt("../data/SimBac/genomes_bool.csv", delimiter=",", dtype=bool)

In [3]:
genome_simbac.shape

(10, 100000)

In [4]:
np.random.seed(100)
clonal_tree = ClonalTree(n=10)

# Load phylo tree and convert to ClonalTree format
phylo_tree = Phylo.read("../data/SimBac/clonal_frame.nwk", "newick")
Phylo.draw_ascii(phylo_tree)

edge, node_height = newick_to_tree(phylo_tree)
clonal_tree.edge = edge
clonal_tree.node_height = node_height
clonal_tree.height = np.max(node_height)
clonal_tree.length = np.sum(edge[:, 2])

                                                                   _______ 2
                                        __________________________|
                            ___________|                          |_______ 8
                           |           |
                           |           |___________________________________ 1
                           |
                        ___|                         _____________________ 6
                       |   |                       ,|
                       |   |                       ||  ___________________ 3
                       |   |                       ||_|
                       |   |_______________________|  |___________________ 9
  _____________________|                           |
 |                     |                           |         _____________ 5
 |                     |                           |________|
_|                     |                                    |_____________ 7
 |                  

## Summary statistics

This notebook evaluates the proportions and exact values of LD and G4 for comparison and selection. The items used in the summary statistics is listed below. Let $d$ denote the distance between sites in a pair, and $L$ denote the length of the gene segment.

1. $\text{LD}_{d < L/2}$

2. $\text{LD}_{d \geq L/2}$

3. $\# \text{G4}_{d < L/2}$

4. $\# \text{G4}_{d \geq L/2}$

5. $\frac{\text{LD}_{d < L/2}}{\# \text{segregating pairs that } d < L/2}$

6. $\frac{\text{LD}_{d \geq L/2}}{\# \text{segregating pairs that } d \geq L/2}$

7. $\frac{\# \text{G4}_{d < L/2}}{\# \text{segregating pairs that } d < L/2}$

8. $\frac{\# \text{G4}_{d \geq L/2}}{\# \text{segregating pairs that } d \geq L/2}$

9. $\text{LD}_{20 \leq d < 50}$

10. $\text{LD}_{50 \leq d \leq 80}$

11. $\# \text{G4}_{20 \leq d < 50}$

12. $\# \text{G4}_{50 \leq d \leq 80}$

13. $\frac{\text{LD}_{20 \leq d < 50}}{\# \text{segregating pairs that } 20 \leq d < 50}$

14. $\frac{\text{LD}_{50 \leq d \leq 80}}{\# \text{segregating pairs that } 50 \leq d \leq 80}$

15. $\frac{\# \text{G4}_{20 \leq d < 50}}{\# \text{segregating pairs that } 20 \leq d < 50}$

16. $\frac{\# \text{G4}_{50 \leq d \leq 80}}{\# \text{segregating pairs that } 50 \leq d \leq 80}$

17. homoplasy index based on the clonal structure

18. Proportion of segregating sites

19. L

### Get summary statistics from SimBac data

In [6]:
x_o_500 = np.full((100, 19), np.nan)
x_o_2000 = np.full((100, 19), np.nan)
x_o_6000 = np.full((100, 19), np.nan)

x_o_500, x_o_2000, x_o_6000

(array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(100, 19)),
 array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(100, 19)),
 array([[nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        ...,
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan],
        [nan, nan, nan, ..., nan, nan, nan]], shape=(100, 19)))

### Randomly choose segments of length 500 bp

In [7]:
np.random.seed(100)
L = 500

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    ld_20_50 += LD_r(mat[:, idx_pair])
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    ld_50_80 += LD_r(mat[:, idx_pair])
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_500[idx, 0] = ld_near
        x_o_500[idx, 1] = ld_far
        x_o_500[idx, 2] = g4_near
        x_o_500[idx, 3] = g4_far

        x_o_500[idx, 4] = ld_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 5] = ld_far /seg_far if seg_far > 0 else 0
        x_o_500[idx, 6] = g4_near / seg_near if seg_near > 0 else 0
        x_o_500[idx, 7] = g4_far / seg_far if seg_far > 0 else 0

        x_o_500[idx, 8] = ld_20_50
        x_o_500[idx, 9] = ld_50_80
        x_o_500[idx, 10] = g4_20_50
        x_o_500[idx, 11] = g4_50_80

        x_o_500[idx, 12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_500[idx, 14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_500[idx, 15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_500[idx, :16] = 0

    # Summary statistic homoplasy index
    x_o_500[idx, 16] = homoplasy_index(clonal_tree, mat)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_500[idx, 17] = count_S / L

    # Add the length of sequence as a summary statistic
    x_o_500[idx, 18] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=500")

x_o_500

Completed 10 out of 100 simulations for L=500
Completed 20 out of 100 simulations for L=500
Completed 30 out of 100 simulations for L=500
Completed 40 out of 100 simulations for L=500
Completed 50 out of 100 simulations for L=500
Completed 60 out of 100 simulations for L=500
Completed 70 out of 100 simulations for L=500
Completed 80 out of 100 simulations for L=500
Completed 90 out of 100 simulations for L=500
Completed 100 out of 100 simulations for L=500


array([[1.98870236e+02, 5.23110769e+01, 1.65000000e+02, ...,
        4.60784314e-01, 1.10000000e-01, 5.00000000e+02],
       [1.42857041e+02, 2.99679967e+01, 4.70000000e+01, ...,
        4.07894737e-01, 9.00000000e-02, 5.00000000e+02],
       [2.32192834e+02, 6.47284895e+01, 1.00000000e+01, ...,
        4.38202247e-01, 1.00000000e-01, 5.00000000e+02],
       ...,
       [2.67573222e+02, 4.89370866e+01, 1.98000000e+02, ...,
        4.76923077e-01, 1.36000000e-01, 5.00000000e+02],
       [1.20851553e+02, 2.01103395e+01, 1.80000000e+01, ...,
        1.57894737e-01, 9.60000000e-02, 5.00000000e+02],
       [2.53144010e+02, 7.06702767e+01, 1.58000000e+02, ...,
        3.16831683e-01, 1.38000000e-01, 5.00000000e+02]], shape=(100, 19))

### Randomly choose a segment of length 2000 bp

In [8]:
np.random.seed(100)
L = 2000

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    ld_20_50 += LD_r(mat[:, idx_pair])
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    ld_50_80 += LD_r(mat[:, idx_pair])
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_2000[idx, 0] = ld_near
        x_o_2000[idx, 1] = ld_far
        x_o_2000[idx, 2] = g4_near
        x_o_2000[idx, 3] = g4_far

        x_o_2000[idx, 4] = ld_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 5] = ld_far /seg_far if seg_far > 0 else 0
        x_o_2000[idx, 6] = g4_near / seg_near if seg_near > 0 else 0
        x_o_2000[idx, 7] = g4_far / seg_far if seg_far > 0 else 0

        x_o_2000[idx, 8] = ld_20_50
        x_o_2000[idx, 9] = ld_50_80
        x_o_2000[idx, 10] = g4_20_50
        x_o_2000[idx, 11] = g4_50_80

        x_o_2000[idx, 12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_2000[idx, 14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_2000[idx, 15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_2000[idx, :16] = 0

    # Summary statistic homoplasy index
    x_o_2000[idx, 16] = homoplasy_index(clonal_tree, mat)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_2000[idx, 17] = count_S / L

    # Add the length of sequence as a summary statistic
    x_o_2000[idx, 18] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=2000")

x_o_2000

Completed 10 out of 100 simulations for L=2000
Completed 20 out of 100 simulations for L=2000
Completed 30 out of 100 simulations for L=2000
Completed 40 out of 100 simulations for L=2000
Completed 50 out of 100 simulations for L=2000
Completed 60 out of 100 simulations for L=2000
Completed 70 out of 100 simulations for L=2000
Completed 80 out of 100 simulations for L=2000
Completed 90 out of 100 simulations for L=2000
Completed 100 out of 100 simulations for L=2000


array([[2.86590447e+03, 7.56347586e+02, 2.92900000e+03, ...,
        4.22222222e-01, 1.17000000e-01, 2.00000000e+03],
       [1.79937694e+03, 6.44566624e+02, 1.05700000e+03, ...,
        3.23129252e-01, 9.95000000e-02, 2.00000000e+03],
       [3.24820851e+03, 7.15723396e+02, 2.05700000e+03, ...,
        4.33734940e-01, 1.17500000e-01, 2.00000000e+03],
       ...,
       [2.22031694e+03, 7.26673872e+02, 2.17500000e+03, ...,
        3.83977901e-01, 1.11500000e-01, 2.00000000e+03],
       [3.40949160e+03, 9.63894802e+02, 1.62400000e+03, ...,
        3.12834225e-01, 1.28500000e-01, 2.00000000e+03],
       [3.03742866e+03, 6.19953458e+02, 3.66300000e+03, ...,
        3.91752577e-01, 1.18000000e-01, 2.00000000e+03]], shape=(100, 19))

### Randomly choose a segment of length 6000 bp

In [ ]:
np.random.seed(100)
L = 6000

for idx in range(100):
    start_pos = np.random.randint(1, genome_simbac.shape[1]-L+1)
    mat = genome_simbac[:, start_pos-1:start_pos+L-1]

    # Identify segregating sites
    has_true = mat.any(axis=0)
    has_false = ~mat.all(axis=0)
    idx_seg = np.where(has_true & has_false)[0]

    # Summary statistics LD r and G4 test
    seg_near, seg_far = 0, 0
    seg_20_50, seg_50_80 = 0, 0
    ld_near, ld_far, g4_near, g4_far = 0, 0, 0, 0
    ld_20_50, ld_50_80, g4_20_50, g4_50_80 = 0, 0, 0, 0
    if idx_seg.size >= 2:
        for i in range(idx_seg.size - 1):
            for j in range(i + 1, idx_seg.size):
                dist_ij = idx_seg[j] - idx_seg[i]
                idx_pair = [idx_seg[i], idx_seg[j]]
                if dist_ij < L/2:
                    ld_near += LD_r(mat[:, idx_pair])
                    g4_near += G4_test(mat[:, idx_pair])
                    seg_near += 1
                else:
                    ld_far += LD_r(mat[:, idx_pair])
                    g4_far += G4_test(mat[:, idx_pair])
                    seg_far += 1
                if 20 <= dist_ij < 50:
                    ld_20_50 += LD_r(mat[:, idx_pair])
                    g4_20_50 += G4_test(mat[:, idx_pair])
                    seg_20_50 += 1
                if 50 <= dist_ij <= 80:
                    ld_50_80 += LD_r(mat[:, idx_pair])
                    g4_50_80 += G4_test(mat[:, idx_pair])
                    seg_50_80 += 1
        
        x_o_6000[idx, 0] = ld_near
        x_o_6000[idx, 1] = ld_far
        x_o_6000[idx, 2] = g4_near
        x_o_6000[idx, 3] = g4_far

        x_o_6000[idx, 4] = ld_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 5] = ld_far /seg_far if seg_far > 0 else 0
        x_o_6000[idx, 6] = g4_near / seg_near if seg_near > 0 else 0
        x_o_6000[idx, 7] = g4_far / seg_far if seg_far > 0 else 0

        x_o_6000[idx, 8] = ld_20_50
        x_o_6000[idx, 9] = ld_50_80
        x_o_6000[idx, 10] = g4_20_50
        x_o_6000[idx, 11] = g4_50_80

        x_o_6000[idx, 12] = ld_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 13] = ld_50_80 / seg_50_80 if seg_50_80 > 0 else 0
        x_o_6000[idx, 14] = g4_20_50 / seg_20_50 if seg_20_50 > 0 else 0
        x_o_6000[idx, 15] = g4_50_80 / seg_50_80 if seg_50_80 > 0 else 0
    else:
        x_o_6000[idx, :16] = 0

    # Summary statistic homoplasy index
    x_o_6000[idx, 16] = homoplasy_index(clonal_tree, mat)

    # Summary statistic proportion of segregating sites
    count_S = idx_seg.size
    x_o_6000[idx, 17] = count_S / L

    # Add the length of sequence as a summary statistic
    x_o_6000[idx, 18] = L

    if (idx + 1) % 10 == 0:
        print(f"Completed {idx + 1} out of 100 simulations for L=6000")

x_o_6000

In [ ]:
x_o_500_mat = x_o_500.copy()
x_o_2000_mat = x_o_2000.copy()
x_o_6000_mat = x_o_6000.copy()

x_o_500_mat, x_o_2000_mat, x_o_6000_mat

(array([2.12980173e-03, 1.66728532e-03, 1.76706827e-03, 1.56175299e-03,
        5.17241379e-02, 1.10000000e-01, 5.00000000e+02]),
 array([1.20078541e-03, 1.28784540e-03, 7.05372039e-04, 9.47052947e-04,
        2.45098039e-02, 9.95000000e-02, 2.00000000e+03]),
 array([1.95249691e-03, 1.65602094e-03, 1.70160424e-03, 2.25635899e-03,
        1.61943320e-02, 1.21500000e-01, 6.00000000e+03]))

In [ ]:
x_o_500 = torch.tensor(x_o_500, device=torch_device)
x_o_500 = x_o_500.to(torch.float32)

x_o_2000 = torch.tensor(x_o_2000, device=torch_device)
x_o_2000 = x_o_2000.to(torch.float32)

x_o_6000 = torch.tensor(x_o_6000, device=torch_device)
x_o_6000 = x_o_6000.to(torch.float32)

In [ ]:
x_o_2000.dtype, x_o_500.dtype, x_o_6000.dtype

(torch.float32, torch.float32, torch.float32)